In [1]:
from utils import *
import wandb

In [2]:
config = Config().load(os.path.join("configs", "vit.json"))

In [3]:
def itertoolsBetter(dataIter):
    while True:
        for batch in dataIter:
            yield batch

In [ ]:
# https://github.com/kreasof-ai/sigreg
def sigreg_weak_loss(x, sketch_dim=64):
    """
    Forces Covariance(x) ~ Identity.
    Matches the 2nd Moment (Spherical Cloud).
    """
    N, C = x.size()
    # 1. Sketching (Optional for C=512, but good for consistency)
    if C > sketch_dim:
        S = torch.randn(sketch_dim, C, device=x.device) / (C ** 0.5)
        x = x @ S.T  # [N, sketch_dim]
    else:
        sketch_dim = C

    # 2. Centering & Covariance
    x = x - x.mean(dim=0, keepdim=True)
    cov = (x.T @ x) / (N - 1 + 1e-6)

    # 3. Target Identity
    target = torch.eye(sketch_dim, device=x.device)

    # 4. Off-diagonal suppression + Diagonal maintenance
    return torch.norm(cov - target, p='fro')


class EmbeddingLoss(nn.Module):
    def __init__(self, temperature=0.01, alpha=0.2):
        super().__init__()
        self.temperature = temperature
        self.alpha = alpha

    def forward(self, x, y):
        B = x.shape[0]
        xNorm, yNorm = nn.functional.normalize(x, dim=-1), nn.functional.normalize(y, dim=-1)

        logits = xNorm @ yNorm.T
        logits = logits / self.temperature

        labels = torch.arange(B, device=x.device)

        loss1 = nn.functional.cross_entropy(logits, labels)
        loss2 = nn.functional.cross_entropy(logits.T, labels)

        infoLoss = 0.5 * (loss1 + loss2)

        sigLoss = sigreg_weak_loss(x) + sigreg_weak_loss(y)

        return infoLoss + self.alpha * sigLoss

In [4]:
def trainModel(config, modelClass, datasetClass):
    dataset = datasetClass(config.dataset)
    model = modelClass(config.model).to(device)

    print(f"Model has {sum([p.numel() for p in model.parameters()])} parameters")

    optimizer = torch.optim.Adam(model.parameters(), lr=config.learningRate)

    objective = EmbeddingLoss()

    train, test = datasetClass.split(dataset, config)
    train = DataLoader(train, batch_size=config.batchSize, shuffle=True, collate_fn=dataset.collate)
    test = DataLoader(test, batch_size=config.batchSize, shuffle=True, collate_fn=dataset.collate)

    testIter = itertoolsBetter(test)

    run = None
    total = 0

    try:
        for epoch in range(config.epochs):
            progress = 0
            for inputsX, inputsY, info in train:
                model.train()
                optimizer.zero_grad()

                _, outputsX = model(inputsX.to(device))
                _, outputsY = model(inputsY.to(device))
                loss = objective(outputsX, outputsY)

                trainLoss = loss.detach().item()

                loss.backward()
                optimizer.step()

                with torch.no_grad():
                    model.eval()
                    inputsX1, inputsY1, info1 = next(testIter)

                    _, outputsX1 = model(inputsX1.to(device))
                    _, outputsY1 = model(inputsY1.to(device))
                    loss1 = objective(outputsX1, outputsY1)

                    testLoss = loss1.detach().item()

                if run is None:
                    run = wandb.init(entity="dylanberndt123-missouri-state-university", project="Briefcase", config=config.serialize())
                
                run.log({"Train Loss": trainLoss, "Test Loss": testLoss}, step=total)

                progress += 1
                total += 1

                print(f"\r{epoch + 1} | {progress}/{len(train)} | {(progress / len(train)) * 100:.3f}% |  Train Loss: {trainLoss:.2f} | Test Loss: {testLoss:.2f}", end="")

    except KeyboardInterrupt:
        latestPath = os.path.join("checkpoints", "pretrain", "latest")
        if not os.path.exists(os.path.join("checkpoints", "pretrain", "latest")):
            os.mkdir(latestPath)

        stamp = datetime.now().strftime("%Y-%m-%d %H-%M")
        timePath = os.path.join("checkpoints", "pretrain", stamp)
        if not os.path.exists(timePath):
            os.mkdir(timePath)

        torch.save(model.state_dict(), os.path.join(latestPath, "checkpoint.pt"))
        torch.save(model.state_dict(), os.path.join(timePath, "checkpoint.pt"))
        config.save(os.path.join(latestPath, "config.json"))
        config.save(os.path.join(timePath, "config.json"))
        return model

In [5]:
trainModel(config, ViT, MyFontsImageData)

Fonts serialized: 1821/3794google\fonts\ofl\kumarone\KumarOne-Regular.ttf execution context too long
Fonts serialized: 2335/3794google\fonts\ofl\notocoloremojicompattest\NotoColorEmojiCompatTest-Regular.ttf invalid pixel size
Fonts serialized: 3702/3794google\fonts\ofl\zcoolxiaowei\ZCOOLXiaoWei-Regular.ttf [Errno 22] Invalid argument: 'google\\bitmaps\\????? ?? al.bmp'
Images loaded: 266201/266219
Model has 9510028 parameters
3 | 208/978 | 21.268% |  Train Loss: 0.12 | Test Loss: 0.000

UNet(
  (input): Sequential(
    (0): Linear(in_features=1, out_features=24, bias=True)
    (1): ReLU()
  )
  (ups): ModuleList(
    (0): Up(
      (conv): ConvBlock(
        (module): Sequential(
          (0): Conv2d(96, 24, kernel_size=(3, 3), stride=(1, 1), padding=same)
          (1): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU()
          (3): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=same)
          (4): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (5): ReLU()
          (6): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=same)
          (7): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (8): ReLU()
          (9): Conv2d(24, 24, kernel_size=(3, 3), stride=(1, 1), padding=same)
          (10): BatchNorm2d(24, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (11): ReLU()
        )
      )